# Digital Witness — Retail Shoplifting Detection Pipeline

## Cell 1 — Install Dependencies

In [1]:
# Install all required packages.
# The try/except pattern avoids reinstalling if already present,
# making re-runs faster on the training machine.
import subprocess, sys

def pip_install(package):
    """Install a package via pip if import fails."""
    try:
        __import__(package.split('[')[0].replace('-', '_').split('>=')[0].split('==')[0])
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

packages = [
    'opencv-python-headless',
    'numpy',
    'pandas',
    'matplotlib',
    'scikit-learn',
    'Pillow',
    'torch',
    'torchvision',
    'ultralytics>=8.3.0',
    'lapx>=0.5.2',
    'reportlab>=4.0.0',
    'tqdm',
    'seaborn',
]


## Cell 2 — Configuration

In [ ]:
import platform
import random
from pathlib import Path
import torch
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 0 if (platform.system() == 'Windows' or device.type != 'cuda') else 4
PIN_MEMORY  = device.type == 'cuda'

ROOT = Path().resolve()
TRAIN_VIDEOS = ROOT / 'data' / 'videos'

MODELS_DIR   = ROOT / 'models'
FRAMES_DIR   = ROOT / 'frames'
OUTPUTS_DIR  = ROOT / 'outputs' / 'cases'
DATASET_YAML = ROOT / 'data' / 'dataset' / 'data.yaml'
SEQ_DIR      = ROOT / 'data' / 'sequences'

YOLO_BASE      = MODELS_DIR / 'yolo26n.pt'
YOLO26_RETAIL  = MODELS_DIR / 'yolo26_dw_v2.pt'
MOBILENET_SAVE = MODELS_DIR / 'mobilenet_dw.pt'
BILSTM_SAVE    = MODELS_DIR / 'bilstm_dw.pt'

for d in [MODELS_DIR, FRAMES_DIR / 'normal', FRAMES_DIR / 'shoplifting',
          OUTPUTS_DIR, SEQ_DIR / 'normal', SEQ_DIR / 'shoplifting']:
    d.mkdir(parents=True, exist_ok=True)

YOLO_EARLY_STOP_PATIENCE      = 5   # YOLO monitors mAP - can oscillate, 5 is tight but fair
MOBILENET_EARLY_STOP_PATIENCE = 5   # MobileNet monitors val_acc
BILSTM_EARLY_STOP_PATIENCE    = 5   # BiLSTM monitors val_acc

SMOKE_TEST = True    # set False before full training run

EPOCHS_YOLO      = 1  if SMOKE_TEST else 75
EPOCHS_MOBILENET = 1  if SMOKE_TEST else 20
EPOCHS_BILSTM    = 1  if SMOKE_TEST else 30

if SMOKE_TEST:
    BATCH_SIZE = 4
elif device.type == 'cuda':
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH_SIZE = 32 if _vram_gb >= 8 else 24 if _vram_gb >= 6 else 16 if _vram_gb >= 4 else 8
    print(f'GPU VRAM: {_vram_gb:.1f} GB → BATCH_SIZE={BATCH_SIZE}')
else:
    BATCH_SIZE = 8

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, 'Ratios must sum to 1.0'

MOBILENET_INPUT_SIZE  = (224, 224)
MOBILENET_FEATURE_DIM = 1280
YOLO_FEAT_DIM         = 4                                    # YOLO 4-class confidence scores appended per frame
BILSTM_INPUT_DIM      = MOBILENET_FEATURE_DIM + YOLO_FEAT_DIM  # 1284
LR_MOBILENET          = 1e-4
FPS_TARGET            = 6

BEHAVIOR_CLASSES = ['normal', 'shoplifting']

LSTM_SEQ_LEN    = 45
LSTM_STRIDE     = 15
LSTM_HIDDEN_DIM = 256
LSTM_NUM_LAYERS = 2
LSTM_DROPOUT    = 0.3
LR_LSTM         = 5e-4

W_BEHAVIOUR    = 0.40
W_CONCEALMENT  = 0.30
W_POS_MISMATCH = 0.20
W_DURATION     = 0.10
assert abs(W_BEHAVIOUR + W_CONCEALMENT + W_POS_MISMATCH + W_DURATION - 1.0) < 1e-9

THRESHOLD_LOW      = 0.30
THRESHOLD_MEDIUM   = 0.50
THRESHOLD_HIGH     = 0.70
THRESHOLD_CRITICAL = 0.85

YOLO_CLASSES = ['Looking around', 'Picking-Holding', 'normal', 'shoplifting']
PERSON_CLASS_IDS     = {0, 1, 2, 3}
PRODUCT_HELD_IDS     = {1}
CONCEALMENT_IDS      = {0, 3}
SHOPLIFTING_CLASS_ID = 3
CHECKOUT_OCCUPIED_ID = -1
CHECKOUT_VACANT_ID   = -1

print('=' * 50)
print('  DIGITAL WITNESS - CONFIGURATION')
print('=' * 50)
print(f'  Device       : {device}')
print(f'  GPU          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'  SMOKE TEST   : {SMOKE_TEST}')
print(f'  Epochs YOLO  : {EPOCHS_YOLO}')
print(f'  Epochs MNet  : {EPOCHS_MOBILENET}')
print(f'  Epochs BiLSTM: {EPOCHS_BILSTM}')
print(f'  Batch size   : {BATCH_SIZE}')
print(f'  Split        : 70% train / 15% val / 15% test')
print(f'  ROOT         : {ROOT}')
print(f'  TRAIN_VIDEOS : {TRAIN_VIDEOS} (exists: {TRAIN_VIDEOS.exists()})')
print('=' * 50)

GPU VRAM: 8.6 GB → BATCH_SIZE=32
  DIGITAL WITNESS — CONFIGURATION
  Device       : cuda
  GPU          : NVIDIA GeForce RTX 3070 Laptop GPU
  SMOKE TEST   : False
  Epochs YOLO  : 50
  Epochs MNet  : 40
  Epochs BiLSTM: 30
  Batch size   : 32
  Split        : 70% train / 15% val / 15% test
  ROOT         : D:\Santosh\Project_DigitalWitness
  TRAIN_VIDEOS : D:\Santosh\Project_DigitalWitness\data\videos (exists: True)


## Cell 3 — YOLO26n Fine-tuning

Fine-tunes the YOLO26n base model on the 24-class Roboflow retail annotation dataset. The backbone is frozen (freeze=10) so only the detection head learns retail-specific classes. Output: `models/yolo26_dw_v2.pt`

In [ ]:
#  CELL 3 - YOLO26n Fine-tuning 
import shutil
import yaml
from ultralytics import YOLO

# Step 1: Verify base weights exist
if not YOLO_BASE.exists():
    raise FileNotFoundError(
        f'Base YOLO weights not found: {YOLO_BASE}\n'
        'Download yolo26n.pt and place it in the models/ directory.'
    )
print(f'Base weights found: {YOLO_BASE}')

# Step 2: Load and verify data.yaml
if not DATASET_YAML.exists():
    raise FileNotFoundError(
        f'Dataset config not found: {DATASET_YAML}\n'
        'Ensure the Roboflow dataset is extracted to data/dataset/.'
    )

with open(DATASET_YAML, 'r') as f:
    yaml_content = yaml.safe_load(f)

num_classes = yaml_content.get('nc', 0)
if num_classes != 4:
    print(f'WARNING: data.yaml has {num_classes} classes but expected 4. '
          f'Check that the shoplitingvideo+handpocket v9 dataset is being used.')
else:
    print(f'data.yaml verified: {num_classes} classes (Looking around, '
          f'Picking-Holding, normal, shoplifting).')

# Step 3: Patch the path field to absolute path for this machine
yaml_content['path'] = str(DATASET_YAML.parent.resolve())
patched_yaml_path = MODELS_DIR / 'data_patched.yaml'
with open(patched_yaml_path, 'w') as f:
    yaml.dump(yaml_content, f)
print(f'Patched data.yaml written to: {patched_yaml_path}')

# Step 4: Load the base YOLO model
model = YOLO(str(YOLO_BASE))

# freeze=10 freezes the first 10 layers of the YOLO backbone.
# This preserves low-level feature detectors (edges, textures) learned on COCO,
# while allowing the detection head to learn the 4 behaviour classes.
# The 4-class schema is simpler than the 24-class retail schema,
# so fewer epochs are needed and the head converges faster.
print(f'\nStarting YOLO fine-tuning (4-class behaviour dataset)...')
print(f'  Epochs   : {EPOCHS_YOLO}')
print(f'  Batch    : {BATCH_SIZE}')
print(f'  Device   : {device}')
print(f'  Freeze   : 10 backbone layers')

results = model.train(
    data=str(patched_yaml_path),
    epochs=EPOCHS_YOLO,
    imgsz=640,           # matches Roboflow preprocessing (416×416)
    batch=-1,            # auto-selects safe batch size for available VRAM
    freeze=10,

    # label_smoothing=0.1: distributes 10% of probability mass across
    # all classes during training instead of pushing the target class to 1.0.
    # This prevents the model from becoming overconfident (100% outputs)
    # and improves calibration on ambiguous frames (e.g. someone bending
    # to pick up a dropped item vs genuine concealment).
    label_smoothing=0.1,

    # Lower learning rate for head-only fine-tuning:
    # The head is being trained from near-random initialisation on 4 new
    # classes.  A smaller lr0 prevents large gradient updates that would
    # destabilise the frozen backbone weights adjacent to the head.
    lr0=0.005,
    lrf=0.01,
    warmup_epochs=3,

    device=str(device),
    patience=YOLO_EARLY_STOP_PATIENCE,  # stop if mAP flat for 5 epochs
    save=True,
    plots=True,
    workers=0,
    project=str(ROOT / 'runs' / 'yolo_ft'),
    name='exp',
    exist_ok=True,
)

# Step 5: Copy best weights to models/
best_weights = Path(results.save_dir) / 'weights' / 'best.pt'
if best_weights.exists():
    shutil.copy(best_weights, YOLO26_RETAIL)
    print(f'\nBest weights copied to: {YOLO26_RETAIL}')
else:
    # Fallback: last.pt if best.pt wasn't saved (e.g. 1-epoch smoke test)
    last_weights = Path(results.save_dir) / 'weights' / 'last.pt'
    if last_weights.exists():
        shutil.copy(last_weights, YOLO26_RETAIL)
        print(f'Smoke test: last.pt copied to: {YOLO26_RETAIL}')

# Step 6: Print final metrics
try:
    rd = results.results_dict
    print(f'\nFinal mAP50    : {rd.get("metrics/mAP50(B)", "N/A")}')
    print(f'Final mAP50-95 : {rd.get("metrics/mAP50-95(B)", "N/A")}')
except Exception:
    print('Training complete (metrics not available in smoke-test run).')

print(f'\nYOLO fine-tuning complete. Saved to: {YOLO26_RETAIL}')

Base weights found: D:\Santosh\Project_DigitalWitness\models\yolo26n.pt
data.yaml verified: 4 classes (Looking around, Picking-Holding, normal, shoplifting).
Patched data.yaml written to: D:\Santosh\Project_DigitalWitness\models\data_patched.yaml

Starting YOLO fine-tuning (4-class behaviour dataset)...
  Epochs   : 50
  Batch    : 32
  Device   : cuda
  Freeze   : 10 backbone layers
New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.21  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Santosh\Project_DigitalWitness\models\data_pat

## Cell 4 — MobileNetV2 Frame Classifier

Extracts frames from behaviour videos at FPS_TARGET (6fps), then trains MobileNetV2 as a binary classifier (normal / shoplifting) on person crops.

**Split:** 70% train / 15% val / 15% test (stratified)
The test set is held out completely and only used in Cell 6 for final reporting.

In [ ]:
import cv2
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import MobileNet_V2_Weights
from sklearn.model_selection import train_test_split
from tqdm import tqdm

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


def extract_frames(video_root, output_dir, classes, fps_target=FPS_TARGET):
    counts = {}
    for cls in classes:
        out_cls_dir = output_dir / cls
        out_cls_dir.mkdir(parents=True, exist_ok=True)

        videos = (list(video_root.glob(f'{cls}/*.mp4')) +
                  list(video_root.glob(f'{cls}/*.avi')) +
                  list(video_root.glob(f'*/{cls}/*.mp4')) +
                  list(video_root.glob(f'*/{cls}/*.avi')))

        if not videos:
            print(f'  WARNING: No videos found for class "{cls}" under {video_root}')
            counts[cls] = 0
            continue

        frame_count = 0
        for vid_path in tqdm(videos, desc=f'Extracting {cls}'):
            cap = cv2.VideoCapture(str(vid_path))
            if not cap.isOpened():
                print(f'  WARNING: Cannot open video: {vid_path}')
                continue

            src_fps = cap.get(cv2.CAP_PROP_FPS)
            step = max(1, int(round(src_fps / fps_target)))

            frame_idx = 0
            saved_idx = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                if frame_idx % step == 0:
                    out_path = out_cls_dir / f'{vid_path.stem}_frame_{saved_idx:05d}.jpg'
                    cv2.imwrite(str(out_path), frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
                    saved_idx += 1
                    frame_count += 1
                frame_idx += 1
            cap.release()

        counts[cls] = frame_count
        print(f'  {cls}: {frame_count} frames extracted')

    return counts


normal_frames = list((FRAMES_DIR / 'normal').glob('*.jpg'))
shop_frames   = list((FRAMES_DIR / 'shoplifting').glob('*.jpg'))

if len(normal_frames) == 0 and len(shop_frames) == 0:
    print('Extracting frames from videos...')
    if TRAIN_VIDEOS.exists():
        counts = extract_frames(TRAIN_VIDEOS, FRAMES_DIR, ['normal', 'shoplifting'])
    else:
        print(f'WARNING: TRAIN_VIDEOS not found at {TRAIN_VIDEOS}')
        counts = {'normal': 0, 'shoplifting': 0}
else:
    counts = {'normal': len(normal_frames), 'shoplifting': len(shop_frames)}
    print(f'Frames already extracted: normal={counts["normal"]}, shoplifting={counts["shoplifting"]}')


class FrameDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(str(self.paths[idx]))
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(MOBILENET_INPUT_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(MOBILENET_INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

all_paths  = []
all_labels = []

for label_idx, cls_name in enumerate(BEHAVIOR_CLASSES):
    cls_frames = sorted((FRAMES_DIR / cls_name).glob('*.jpg'))
    all_paths.extend(cls_frames)
    all_labels.extend([label_idx] * len(cls_frames))

print(f'Total frames available: {len(all_paths)}')
if len(all_paths) == 0:
    print('WARNING: No frames found. Skipping training.')
else:
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        all_paths, all_labels,
        test_size=TEST_RATIO,
        stratify=all_labels,
        random_state=42,
    )

    val_size_adjusted = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval,
        test_size=val_size_adjusted,
        stratify=y_trainval,
        random_state=42,
    )

    test_split = {'paths': [str(p) for p in X_test], 'labels': y_test}
    with open(MODELS_DIR / 'mobilenet_test_split.json', 'w') as f:
        json.dump(test_split, f)

    def class_dist(labels):
        c = Counter(labels)
        total = sum(c.values())
        return f'{c[0]/total*100:.0f}% normal / {c[1]/total*100:.0f}% shoplifting'

    print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')
    print(f'  Train dist: {class_dist(y_train)}')

    class_counts   = Counter(y_train)
    class_weights  = {c: 1.0 / count for c, count in class_counts.items()}
    sample_weights = [class_weights[label] for label in y_train]
    sampler        = WeightedRandomSampler(sample_weights, len(y_train), replacement=True)

    loss_weights = torch.tensor(
        [class_weights[0], class_weights[1]], dtype=torch.float32
    ).to(device)
    criterion = nn.CrossEntropyLoss(weight=loss_weights)

    train_ds = FrameDataset(X_train, y_train, transform=train_transform)
    val_ds   = FrameDataset(X_val,   y_val,   transform=val_transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


    class MobileNetV2Classifier(nn.Module):
        def __init__(self, num_classes=2):
            super().__init__()
            base = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
            self.features = base.features
            for i, layer in enumerate(self.features):
                if i <= 14:
                    for param in layer.parameters():
                        param.requires_grad = False
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            self.classifier = nn.Sequential(
                nn.Linear(MOBILENET_FEATURE_DIM, 512),
                nn.ReLU(inplace=True),
                nn.Dropout(p=0.3),
                nn.Linear(512, num_classes),
            )

        def forward(self, x):
            x = self.features(x)
            x = self.pool(x)
            x = torch.flatten(x, 1)
            return self.classifier(x)

        def extract_features(self, x):
            x = self.features(x)
            x = self.pool(x)
            return torch.flatten(x, 1)


    model = MobileNetV2Classifier(num_classes=2).to(device)

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_MOBILENET,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.5)

    best_val_acc   = 0.0
    patience_count = 0

    print('\nStarting MobileNetV2 training...')
    print(f'{"Epoch":>5}  {"Train Loss":>10}  {"Train Acc":>9}  {"Val Acc":>7}  {"LR":>8}')
    print('-' * 50)

    for epoch in range(EPOCHS_MOBILENET):
        model.train()
        train_loss    = 0.0
        train_correct = 0
        train_total   = 0

        for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS_MOBILENET}', leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss    += loss.item() * imgs.size(0)
            preds          = outputs.argmax(1)
            train_correct += (preds == labels).sum().item()
            train_total   += imgs.size(0)

        train_acc = train_correct / max(train_total, 1)
        avg_loss  = train_loss    / max(train_total, 1)

        model.eval()
        val_correct  = 0
        val_total    = 0
        val_loss_sum = 0.0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs       = model(imgs)
                loss          = criterion(outputs, labels)
                val_loss_sum += loss.item() * imgs.size(0)
                preds         = outputs.argmax(1)
                val_correct  += (preds == labels).sum().item()
                val_total    += imgs.size(0)

        val_acc    = val_correct / max(val_total, 1)
        current_lr = scheduler.get_last_lr()[0]

        print(f'{epoch+1:>5}  {avg_loss:>10.4f}  {train_acc*100:>8.1f}%  '
              f'{val_acc*100:>6.1f}%  {current_lr:>8.6f}')

        if val_acc > best_val_acc:
            best_val_acc   = val_acc
            patience_count = 0
            torch.save({
                'state_dict':     model.state_dict(),
                'val_acc':        best_val_acc,
                'epochs_trained': epoch + 1,
                'feature_dim':    MOBILENET_FEATURE_DIM,
                'classes':        BEHAVIOR_CLASSES,
            }, MOBILENET_SAVE)
        else:
            patience_count += 1

        scheduler.step()

        if patience_count >= MOBILENET_EARLY_STOP_PATIENCE:
            print(f'\nEarly stopping at epoch {epoch+1}.')
            break

    print(f'\nMobileNetV2 training complete. Best val acc: {best_val_acc*100:.1f}%')
    print(f'Saved to: {MOBILENET_SAVE}')

    # Load best checkpoint for metrics
    ckpt = torch.load(MOBILENET_SAVE, map_location=device)
    model.load_state_dict(ckpt['state_dict'])

    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                  f1_score, confusion_matrix, classification_report)

    val_preds_all = []
    val_true_all  = []
    model.eval()
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs  = imgs.to(device)
            preds = model(imgs).argmax(1).cpu().numpy()
            val_preds_all.extend(preds.tolist())
            val_true_all.extend(labels.numpy().tolist())

    acc  = accuracy_score(val_true_all, val_preds_all)
    prec = precision_score(val_true_all, val_preds_all, average='weighted', zero_division=0)
    rec  = recall_score(val_true_all, val_preds_all, average='weighted', zero_division=0)
    f1   = f1_score(val_true_all, val_preds_all, average='weighted', zero_division=0)
    cm   = confusion_matrix(val_true_all, val_preds_all)
    cr   = classification_report(val_true_all, val_preds_all,
                                  target_names=BEHAVIOR_CLASSES, zero_division=0)

    print('\n' + '=' * 50)
    print('  MobileNetV2 - Validation Metrics')
    print(f'  Accuracy  : {acc*100:.2f}%')
    print(f'  Precision : {prec*100:.2f}%')
    print(f'  Recall    : {rec*100:.2f}%')
    print(f'  F1 Score  : {f1*100:.2f}%')
    print('=' * 50)
    print(cr)

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=BEHAVIOR_CLASSES, yticklabels=BEHAVIOR_CLASSES, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title('MobileNetV2 - Confusion Matrix (Validation)')
    plt.tight_layout()
    plt.show()

Frames already extracted: normal=32050, shoplifting=61575
Total frames available: 93625
Train: 65537 | Val: 14044 | Test: 14044
  Train dist: 34% normal / 66% shoplifting

Starting MobileNetV2 training...
Epoch  Train Loss  Train Acc  Val Acc        LR
--------------------------------------------------


    1      0.0434      97.9%    99.0%  0.000100


    2      0.0160      99.3%    99.5%  0.000100


    3      0.0116      99.5%    99.5%  0.000100


    4      0.0099      99.6%    99.4%  0.000100


...

   14      0.0051      99.8%    99.6%  0.000050


   15      0.0054      99.8%    99.6%  0.000050

Early stopping at epoch 15.

MobileNetV2 training complete. Best val acc: 99.6%
Saved to: D:\Santosh\Project_DigitalWitness\models\mobilenet_dw.pt


C:\Users\johnf\AppData\Local\Temp\ipykernel_19488\3670334501.py:286: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(MOBILENET_SAVE, map_location=device)



  MobileNetV2 — Validation Metrics
  Accuracy  : 99.64%
  Precision : 99.65%
  Recall    : 99.64%
  F1 Score  : 99.64%
              precision    recall  f1-score   support

      normal       0.99      1.00      0.99      4808
 shoplifting       1.00      1.00      1.00      9236

    accuracy                           1.00     14044
   macro avg       1.00      1.00      1.00     14044
weighted avg       1.00      1.00      1.00     14044



<Figure size 500x400 with 2 Axes>

## Cell 5 — BiLSTM + Temporal Attention Classifier

Trains a Bidirectional LSTM with a learned attention mechanism on sequences of MobileNetV2 feature vectors extracted from behaviour videos.

This is the core temporal reasoning component of Digital Witness. A single frame cannot distinguish intentional concealment from legitimate bag repacking. The BiLSTM analyses 7.5 seconds (45 frames) of context, and the attention mechanism learns which specific frames drove the classification. Those attention weights are the XAI (Explainable AI) output of the system.

**Split:** Same 70/15/15 strategy. Test set held out until Cell 7.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import MobileNet_V2_Weights
from sklearn.model_selection import train_test_split
from collections import Counter
from tqdm import tqdm
from ultralytics import YOLO as _YOLO

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


def extract_features_from_video(video_path, mobilenet_model, yolo_model=None, fps_target=FPS_TARGET):
    import cv2
    infer_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize(MOBILENET_INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return np.zeros((1, BILSTM_INPUT_DIM), dtype=np.float32)

    src_fps  = cap.get(cv2.CAP_PROP_FPS)
    step     = max(1, int(round(src_fps / fps_target)))
    features = []

    frame_idx = 0
    mobilenet_model.eval()
    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % step == 0:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                # MobileNetV2: 1280-dim spatial appearance features
                tensor    = infer_transform(frame_rgb).unsqueeze(0).to(device)
                mnet_feat = mobilenet_model.extract_features(tensor).cpu().numpy()[0]

                # YOLO: 4-dim per-frame behaviour confidence scores
                # Index: 0=Looking around, 1=Picking-Holding, 2=normal, 3=shoplifting
                # For each class take the max confidence across all detections in this frame.
                yolo_conf = np.zeros(4, dtype=np.float32)
                if yolo_model is not None:
                    yolo_res = yolo_model(frame, verbose=False, conf=0.1, imgsz=320)
                    if yolo_res and len(yolo_res[0].boxes) > 0:
                        for box in yolo_res[0].boxes:
                            cls_id = int(box.cls[0])
                            conf   = float(box.conf[0])
                            if 0 <= cls_id < 4:
                                yolo_conf[cls_id] = max(yolo_conf[cls_id], conf)

                # Combined: 1280 (appearance) + 4 (behaviour) = 1284-dim
                features.append(np.concatenate([mnet_feat, yolo_conf]))
            frame_idx += 1
    cap.release()

    if not features:
        return np.zeros((1, BILSTM_INPUT_DIM), dtype=np.float32)
    return np.array(features, dtype=np.float32)


# Load MobileNetV2
if MOBILENET_SAVE.exists():
    from torchvision import models as tv_models

    class _MNetV2(nn.Module):
        def __init__(self):
            super().__init__()
            base = tv_models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
            self.features   = base.features
            self.pool       = nn.AdaptiveAvgPool2d((1, 1))
            self.classifier = nn.Sequential(
                nn.Linear(MOBILENET_FEATURE_DIM, 512),
                nn.ReLU(inplace=True),
                nn.Dropout(p=0.3),
                nn.Linear(512, 2),
            )
        def forward(self, x):
            x = self.features(x)
            x = self.pool(x)
            return self.classifier(torch.flatten(x, 1))
        def extract_features(self, x):
            x = self.features(x)
            x = self.pool(x)
            return torch.flatten(x, 1)

    mnet = _MNetV2().to(device)
    ckpt = torch.load(MOBILENET_SAVE, map_location=device)
    sd   = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt
    mnet.load_state_dict(sd)
    mnet.eval()
    print(f'MobileNetV2 loaded from {MOBILENET_SAVE}')
else:
    print(f'WARNING: {MOBILENET_SAVE} not found. Run Cell 4 first.')
    mnet = None

# Load fine-tuned YOLO for behaviour score augmentation
yolo_feat_model = None
_yolo_path = YOLO26_RETAIL if YOLO26_RETAIL.exists() else (YOLO26_DW_V2 if YOLO26_DW_V2.exists() else None)
if _yolo_path is not None:
    yolo_feat_model = _YOLO(str(_yolo_path))
    print(f'YOLO behaviour augmentation: {_yolo_path.name}')
else:
    print('WARNING: No fine-tuned YOLO found. YOLO scores will be zeros.')
    print('Run Cell 3 first for best results.')

# Extract and cache sequences as .npy files
# Skips files that already exist at the correct dimension (BILSTM_INPUT_DIM).
# Replaces files saved with the old 1280-dim format.
if mnet is not None:
    for cls_name in BEHAVIOR_CLASSES:
        out_cls_dir = SEQ_DIR / cls_name
        out_cls_dir.mkdir(parents=True, exist_ok=True)

        vids = (list(TRAIN_VIDEOS.glob(f'{cls_name}/*.mp4')) +
                list(TRAIN_VIDEOS.glob(f'{cls_name}/*.avi')) +
                list(TRAIN_VIDEOS.glob(f'*/{cls_name}/*.mp4')) +
                list(TRAIN_VIDEOS.glob(f'*/{cls_name}/*.avi')))

        for vid in tqdm(vids, desc=f'Extracting sequences: {cls_name}'):
            npy_path = out_cls_dir / f'{vid.stem}.npy'
            if npy_path.exists():
                existing = np.load(str(npy_path), mmap_mode='r')
                if existing.shape[1] == BILSTM_INPUT_DIM:
                    continue  # already at correct dimension - skip
            seq = extract_features_from_video(vid, mnet, yolo_feat_model)
            np.save(str(npy_path), seq)

    print('Sequence extraction complete.')


class SequenceDataset(Dataset):
    def __init__(self, seq_paths, labels, seq_len=LSTM_SEQ_LEN, stride=LSTM_STRIDE):
        self.seq_len = seq_len
        self.stride  = stride
        self.index       = []
        self.labels_list = []
        for i, (p, lbl) in enumerate(zip(seq_paths, labels)):
            seq = np.load(str(p))
            T   = seq.shape[0]
            starts = list(range(0, max(1, T - seq_len + 1), stride))
            self.index.extend([(p, s) for s in starts])
            self.labels_list.extend([lbl] * len(starts))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        path, start = self.index[idx]
        label       = self.labels_list[idx]
        seq    = np.load(str(path))
        window = seq[start: start + self.seq_len]
        if window.shape[0] < self.seq_len:
            pad    = np.zeros((self.seq_len - window.shape[0], BILSTM_INPUT_DIM), dtype=np.float32)
            window = np.concatenate([window, pad], axis=0)
        return torch.tensor(window, dtype=torch.float32), label


seq_paths_all  = []
seq_labels_all = []

for label_idx, cls_name in enumerate(BEHAVIOR_CLASSES):
    files = sorted((SEQ_DIR / cls_name).glob('*.npy'))
    seq_paths_all.extend(files)
    seq_labels_all.extend([label_idx] * len(files))

print(f'Total sequences: {len(seq_paths_all)}')

if len(seq_paths_all) > 0:
    X_sv, X_stest, y_sv, y_stest = train_test_split(
        seq_paths_all, seq_labels_all,
        test_size=TEST_RATIO, stratify=seq_labels_all, random_state=42,
    )
    val_adj = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    X_strain, X_sval, y_strain, y_sval = train_test_split(
        X_sv, y_sv, test_size=val_adj, stratify=y_sv, random_state=42,
    )

    bilstm_test_split = {'paths': [str(p) for p in X_stest], 'labels': y_stest}
    with open(MODELS_DIR / 'bilstm_test_split.json', 'w') as f:
        json.dump(bilstm_test_split, f)
    print(f'BiLSTM test split saved: {len(X_stest)} sequences')

    train_seq_ds = SequenceDataset(X_strain, y_strain)
    val_seq_ds   = SequenceDataset(X_sval,   y_sval)

    seq_class_counts  = Counter(y_strain)
    seq_class_weights = {c: 1.0 / cnt for c, cnt in seq_class_counts.items()}
    seq_sample_w = [seq_class_weights[lbl] for lbl in train_seq_ds.labels_list]
    seq_sampler  = WeightedRandomSampler(seq_sample_w, len(seq_sample_w), replacement=True)

    seq_loss_w = torch.tensor(
        [seq_class_weights[0], seq_class_weights[1]], dtype=torch.float32
    ).to(device)
    seq_criterion = nn.CrossEntropyLoss(weight=seq_loss_w)

    seq_train_loader = DataLoader(train_seq_ds, batch_size=BATCH_SIZE, sampler=seq_sampler,
                                   num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    seq_val_loader   = DataLoader(val_seq_ds,   batch_size=BATCH_SIZE, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, lstm_output):
        scores  = self.attn(lstm_output).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, weights


class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, feature_dim=BILSTM_INPUT_DIM,
                 hidden_dim=LSTM_HIDDEN_DIM, num_layers=LSTM_NUM_LAYERS,
                 num_classes=2, dropout=LSTM_DROPOUT):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention  = TemporalAttention(hidden_dim * 2)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        lstm_out, _           = self.bilstm(x)
        context, attn_weights = self.attention(lstm_out)
        out = self.dropout(context)
        return self.classifier(out), attn_weights


if len(seq_paths_all) > 0:
    bilstm_model = BiLSTMAttentionClassifier().to(device)
    bilstm_opt   = torch.optim.Adam(bilstm_model.parameters(), lr=LR_LSTM)
    bilstm_sched = torch.optim.lr_scheduler.StepLR(bilstm_opt, step_size=8, gamma=0.5)

    best_val_acc_lstm = 0.0
    patience_lstm     = 0
    EARLY_STOP_PAT    = BILSTM_EARLY_STOP_PATIENCE

    print('Starting BiLSTM training...')
    print(f'  Input dim : {BILSTM_INPUT_DIM}  ({MOBILENET_FEATURE_DIM} MobileNetV2 + {YOLO_FEAT_DIM} YOLO)')
    print(f'{"Epoch":>5}  {"Train Loss":>10}  {"Train Acc":>9}  {"Val Acc":>7}')
    print('-' * 45)

    for epoch in range(EPOCHS_BILSTM):
        bilstm_model.train()
        t_loss    = 0.0
        t_correct = 0
        t_total   = 0

        for seqs, labels in tqdm(seq_train_loader, desc=f'BiLSTM Epoch {epoch+1}/{EPOCHS_BILSTM}', leave=False):
            seqs, labels = seqs.to(device), labels.to(device)
            bilstm_opt.zero_grad()
            logits, _ = bilstm_model(seqs)
            loss      = seq_criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(bilstm_model.parameters(), max_norm=5.0)
            bilstm_opt.step()
            t_loss    += loss.item() * seqs.size(0)
            preds      = logits.argmax(1)
            t_correct += (preds == labels).sum().item()
            t_total   += seqs.size(0)

        t_acc = t_correct / max(t_total, 1)
        avg_l = t_loss    / max(t_total, 1)

        bilstm_model.eval()
        v_correct = 0
        v_total   = 0

        with torch.no_grad():
            for seqs, labels in seq_val_loader:
                seqs, labels = seqs.to(device), labels.to(device)
                logits, _    = bilstm_model(seqs)
                preds         = logits.argmax(1)
                v_correct    += (preds == labels).sum().item()
                v_total      += seqs.size(0)

        v_acc = v_correct / max(v_total, 1)
        print(f'{epoch+1:>5}  {avg_l:>10.4f}  {t_acc*100:>8.1f}%  {v_acc*100:>6.1f}%')

        if v_acc > best_val_acc_lstm:
            best_val_acc_lstm = v_acc
            patience_lstm     = 0
            torch.save({
                'state_dict': bilstm_model.state_dict(),
                'val_acc':    best_val_acc_lstm,
                'config': {
                    'feature_dim':   BILSTM_INPUT_DIM,
                    'mobilenet_dim': MOBILENET_FEATURE_DIM,
                    'yolo_feat_dim': YOLO_FEAT_DIM,
                    'hidden_dim':    LSTM_HIDDEN_DIM,
                    'num_layers':    LSTM_NUM_LAYERS,
                    'seq_len':       LSTM_SEQ_LEN,
                    'classes':       BEHAVIOR_CLASSES,
                },
            }, BILSTM_SAVE)
        else:
            patience_lstm += 1

        bilstm_sched.step()

        if patience_lstm >= EARLY_STOP_PAT:
            print(f'Early stopping at epoch {epoch+1}.')
            break

    print(f'\nBiLSTM training complete. Best val acc: {best_val_acc_lstm*100:.1f}%')
    print(f'Saved to: {BILSTM_SAVE}')

    # Load best checkpoint for validation metrics
    bckpt = torch.load(BILSTM_SAVE, map_location=device)
    bilstm_model.load_state_dict(bckpt['state_dict'])

    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                  f1_score, confusion_matrix, classification_report)

    sv_preds = []
    sv_true  = []
    bilstm_model.eval()
    with torch.no_grad():
        for seqs, labels in seq_val_loader:
            seqs      = seqs.to(device)
            logits, _ = bilstm_model(seqs)
            preds     = logits.argmax(1).cpu().numpy()
            sv_preds.extend(preds.tolist())
            sv_true.extend(labels.numpy().tolist())

    b_acc  = accuracy_score(sv_true, sv_preds)
    b_prec = precision_score(sv_true, sv_preds, average='weighted', zero_division=0)
    b_rec  = recall_score(sv_true, sv_preds, average='weighted', zero_division=0)
    b_f1   = f1_score(sv_true, sv_preds, average='weighted', zero_division=0)
    b_cm   = confusion_matrix(sv_true, sv_preds)
    b_cr   = classification_report(sv_true, sv_preds,
                                    target_names=BEHAVIOR_CLASSES, zero_division=0)

    print('\n' + '=' * 50)
    print('  BiLSTM - Validation Metrics')
    print(f'  Accuracy  : {b_acc*100:.2f}%')
    print(f'  Precision : {b_prec*100:.2f}%')
    print(f'  Recall    : {b_rec*100:.2f}%')
    print(f'  F1 Score  : {b_f1*100:.2f}%')
    print('=' * 50)
    print(b_cr)

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(b_cm, annot=True, fmt='d', cmap='Oranges',
                xticklabels=BEHAVIOR_CLASSES, yticklabels=BEHAVIOR_CLASSES, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title('BiLSTM - Confusion Matrix (Validation)')
    plt.tight_layout()
    plt.show()
else:
    print('No sequences found - skipping BiLSTM training. Run Cell 4 first.')

C:\Users\johnf\AppData\Local\Temp\ipykernel_19488\3105453784.py:84: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(MOBILENET_SAVE, map_location=device)


MobileNetV2 loaded from D:\Santosh\Project_DigitalWitness\models\mobilenet_dw.pt


Extracting sequences: shoplifting: 100%|██████████| 134/134 [00:00<00:00, 8375.73it/s]


Sequence feature extraction complete.
Total sequences: 263
BiLSTM test split saved: 40 sequences
Starting BiLSTM training...
Epoch  Train Loss  Train Acc  Val Acc
---------------------------------------------


    1      0.0288      99.2%    99.5%


...

    7      0.0040      99.9%    99.5%


    8      0.0029      99.9%    99.5%
Early stopping at epoch 8.

BiLSTM training complete. Best val acc: 99.8%
Saved to: D:\Santosh\Project_DigitalWitness\models\bilstm_dw.pt


C:\Users\johnf\AppData\Local\Temp\ipykernel_19488\3105453784.py:300: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  bckpt = torch.load(BILSTM_SAVE, map_location=device)



  BiLSTM — Validation Metrics
  Accuracy  : 99.83%
  Precision : 99.83%
  Recall    : 99.83%
  F1 Score  : 99.83%
              precision    recall  f1-score   support

      normal       1.00      1.00      1.00       305
 shoplifting       1.00      1.00      1.00       298

    accuracy                           1.00       603
   macro avg       1.00      1.00      1.00       603
weighted avg       1.00      1.00      1.00       603



<Figure size 500x400 with 2 Axes>

## Cell 6 — YOLO Fine-tuned Model Evaluation

Runs validation on the fine-tuned YOLO model using the patched dataset config.
Reports per-class Precision, Recall, mAP@0.50 and mAP@0.50:0.95, and renders the YOLO confusion matrix.

In [ ]:
import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ultralytics import YOLO

if not YOLO26_RETAIL.exists():
    raise FileNotFoundError(
        f'Fine-tuned YOLO weights not found: {YOLO26_RETAIL}\nRun Cell 3 first.'
    )

patched_yaml_path = MODELS_DIR / 'data_patched.yaml'
if not patched_yaml_path.exists():
    raise FileNotFoundError(
        f'Patched data.yaml not found: {patched_yaml_path}\nRun Cell 3 first.'
    )

print(f'Loading fine-tuned YOLO from: {YOLO26_RETAIL}')
eval_model = YOLO(str(YOLO26_RETAIL))

YOLO_EVAL_DIR = ROOT / 'runs' / 'yolo_eval'

metrics = eval_model.val(
    data=str(patched_yaml_path),
    imgsz=640,
    batch=BATCH_SIZE,
    device=str(device),
    workers=0,
    plots=True,
    save_json=True,
    project=str(YOLO_EVAL_DIR),
    name='eval',
    exist_ok=True,
)

mp      = float(metrics.box.mp)
mr      = float(metrics.box.mr)
map50   = float(metrics.box.map50)
map5095 = float(metrics.box.map)

print('=' * 60)
print('  YOLO - FINE-TUNED MODEL EVALUATION')
print('=' * 60)
print(f'  Precision (mean)  : {mp*100:.2f}%')
print(f'  Recall    (mean)  : {mr*100:.2f}%')
print(f'  mAP @ 0.50        : {map50*100:.2f}%')
print(f'  mAP @ 0.50:0.95   : {map5095*100:.2f}%')
print('=' * 60)

has_per_class = (
    hasattr(metrics.box, 'p') and
    metrics.box.p is not None and
    len(metrics.box.p) == len(YOLO_CLASSES)
)

print(f'\n{"Class":<22} {"Precision":>10} {"Recall":>8} {"AP@50":>8}')
print('-' * 52)
per_class_data = {}
for i, cls_name in enumerate(YOLO_CLASSES):
    if has_per_class:
        p  = float(metrics.box.p[i])
        r  = float(metrics.box.r[i])
        ap = float(metrics.box.ap50[i])
    else:
        p = r = ap = float('nan')
    print(f'{cls_name:<22} {p*100:>9.1f}% {r*100:>7.1f}% {ap*100:>7.1f}%')
    per_class_data[cls_name] = {'precision': p, 'recall': r, 'ap50': ap}

yolo_eval_results = {
    'model':     'YOLO26n_finetuned',
    'precision': mp,
    'recall':    mr,
    'mAP50':     map50,
    'mAP50_95':  map5095,
    'per_class': per_class_data,
}
eval_json_path = MODELS_DIR / 'yolo_eval.json'
with open(eval_json_path, 'w') as f:
    json.dump(yolo_eval_results, f, indent=2)
print(f'\nMetrics saved to: {eval_json_path}')

cm_candidates = [
    YOLO_EVAL_DIR / 'eval' / 'confusion_matrix_normalized.png',
    YOLO_EVAL_DIR / 'eval' / 'confusion_matrix.png',
]
cm_img_path = next((p for p in cm_candidates if p.exists()), None)

if cm_img_path:
    shutil.copy(cm_img_path, OUTPUTS_DIR / 'yolo_confusion_matrix.png')
    img = mpimg.imread(str(cm_img_path))
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('YOLO - Confusion Matrix (Validation)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f'Confusion matrix saved to: {OUTPUTS_DIR / "yolo_confusion_matrix.png"}')
else:
    print('Confusion matrix image not found - run with plots=True on a full (non-smoke-test) dataset.')


Loading fine-tuned YOLO from: D:\Santosh\Project_DigitalWitness\models\yolo26_dw_v2.pt
Ultralytics 8.4.21  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.30.2 ms, read: 3.51.4 MB/s, size: 33.7 KB)
val: Scanning D:\Santosh\Project_DigitalWitness\Data\dataset\valid\labels.cache... 970 images, 47 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 970/970  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 172, len(boxes) = 1264. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 61/61 4.3it/s 14.0s0.2s
                   all        970       1264      0.676      0.645      0.675      0.412
    

<Figure size 800x700 with 1 Axes>

Confusion matrix saved to: D:\Santosh\Project_DigitalWitness\outputs\cases\yolo_confusion_matrix.png


## Cell 7 — BiLSTM Evaluation on Held-Out Test Set + XAI Visualisation

Evaluates the BiLSTM classifier on the held-out test sequences.
Also generates the XAI attention weight plot — showing which frames in a sample sequence drove the classification decision.

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

bilstm_test_path = MODELS_DIR / 'bilstm_test_split.json'
if not bilstm_test_path.exists():
    raise FileNotFoundError(f'BiLSTM test split not found: {bilstm_test_path}\nRun Cell 5 first.')

with open(bilstm_test_path, 'r') as f:
    btest = json.load(f)

X_btest = [Path(p) for p in btest['paths']]
y_btest = btest['labels']
print(f'BiLSTM test split loaded: {len(X_btest)} sequences')


class _TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1),
        )
    def forward(self, lstm_output):
        scores  = self.attn(lstm_output).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, weights


class _BiLSTMEval(nn.Module):
    def __init__(self):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_size=BILSTM_INPUT_DIM,
            hidden_size=LSTM_HIDDEN_DIM,
            num_layers=LSTM_NUM_LAYERS,
            batch_first=True,
            bidirectional=True,
            dropout=LSTM_DROPOUT if LSTM_NUM_LAYERS > 1 else 0.0,
        )
        self.attention  = _TemporalAttention(LSTM_HIDDEN_DIM * 2)
        self.dropout    = nn.Dropout(LSTM_DROPOUT)
        self.classifier = nn.Linear(LSTM_HIDDEN_DIM * 2, 2)
    def forward(self, x):
        lstm_out, _ = self.bilstm(x)
        context, attn_weights = self.attention(lstm_out)
        return self.classifier(self.dropout(context)), attn_weights


if not BILSTM_SAVE.exists():
    raise FileNotFoundError(f'Trained BiLSTM not found: {BILSTM_SAVE}\nRun Cell 5 first.')

bilstm_eval = _BiLSTMEval().to(device)
bckpt = torch.load(BILSTM_SAVE, map_location=device)
bilstm_eval.load_state_dict(bckpt['state_dict'])
bilstm_eval.eval()
print(f'BiLSTM loaded (best val acc: {bckpt["val_acc"]*100:.1f}%)')


class TestSeqDataset(Dataset):
    def __init__(self, paths, labels, seq_len=LSTM_SEQ_LEN):
        self.paths   = paths
        self.labels  = labels
        self.seq_len = seq_len
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        seq    = np.load(str(self.paths[idx]))
        window = seq[:self.seq_len]
        if window.shape[0] < self.seq_len:
            pad    = np.zeros((self.seq_len - window.shape[0], BILSTM_INPUT_DIM))
            window = np.concatenate([window, pad], axis=0)
        return torch.tensor(window, dtype=torch.float32), self.labels[idx]


btest_ds     = TestSeqDataset(X_btest, y_btest)
btest_loader = DataLoader(btest_ds, batch_size=16, shuffle=False, num_workers=NUM_WORKERS)

b_preds             = []
b_true              = []
b_attn_weights_list = []
b_conf_list         = []

with torch.no_grad():
    for seqs, labels in tqdm(btest_loader, desc='Evaluating BiLSTM'):
        seqs        = seqs.to(device)
        logits, atw = bilstm_eval(seqs)
        probs        = torch.softmax(logits, dim=1)
        preds        = logits.argmax(1).cpu().numpy()
        b_preds.extend(preds.tolist())
        b_true.extend(labels.numpy().tolist())
        b_attn_weights_list.extend(atw.cpu().numpy().tolist())
        b_conf_list.extend(probs[:, 1].cpu().numpy().tolist())

b_acc  = accuracy_score(b_true, b_preds)
b_prec = precision_score(b_true, b_preds, average='weighted', zero_division=0)
b_rec  = recall_score(b_true, b_preds, average='weighted', zero_division=0)
b_f1   = f1_score(b_true, b_preds, average='weighted', zero_division=0)
b_cm   = confusion_matrix(b_true, b_preds)
b_cr   = classification_report(b_true, b_preds, target_names=BEHAVIOR_CLASSES, zero_division=0)

print('=' * 60)
print('  BILSTM - TEST SET EVALUATION')
print('=' * 60)
print(f'  Test sequences : {len(b_true)}')
print(f'  Accuracy       : {b_acc*100:.2f}%')
print(f'  Precision      : {b_prec*100:.2f}%')
print(f'  Recall         : {b_rec*100:.2f}%')
print(f'  F1 Score       : {b_f1*100:.2f}%')
print('=' * 60)
print(b_cr)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(b_cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=BEHAVIOR_CLASSES, yticklabels=BEHAVIOR_CLASSES, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('BiLSTM - Confusion Matrix (Test Set)', fontsize=13)
plt.tight_layout()
b_cm_path = OUTPUTS_DIR / 'bilstm_confusion_matrix.png'
plt.savefig(b_cm_path, dpi=150, bbox_inches='tight')
plt.show()

bilstm_eval_results = {
    'model': 'BiLSTM+Attention',
    'test_sequences': len(b_true),
    'accuracy':  float(b_acc),
    'precision': float(b_prec),
    'recall':    float(b_rec),
    'f1_score':  float(b_f1),
    'confusion_matrix': b_cm.tolist(),
}
with open(MODELS_DIR / 'bilstm_eval.json', 'w') as f:
    json.dump(bilstm_eval_results, f, indent=2)
print(f'BiLSTM metrics saved to: {MODELS_DIR / "bilstm_eval.json"}')


def plot_attention_weights(attention_weights, title='Temporal Attention - XAI Output',
                            predicted_class='shoplifting', confidence=0.0):
    fig, ax = plt.subplots(figsize=(12, 4))
    frames = range(len(attention_weights))
    colors = ['#e74c3c' if w > 0.04 else '#2ecc71' for w in attention_weights]
    ax.bar(frames, attention_weights, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(1 / len(attention_weights), color='gray', linestyle='--', linewidth=1,
               label=f'Uniform baseline (1/{len(attention_weights):.0f})')
    ax.set_xlabel('Frame index in 45-frame window (7.5 seconds @ 6fps)', fontsize=11)
    ax.set_ylabel('Attention weight', fontsize=11)
    ax.set_title(f'{title}\nPredicted: {predicted_class.upper()}  |  Confidence: {confidence:.1%}',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    plt.tight_layout()
    xai_path = OUTPUTS_DIR / 'attention_xai_example.png'
    plt.savefig(xai_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'XAI attention plot saved to: {xai_path}')
    return fig


xai_idx = None
for i, (pred, true_lbl, conf) in enumerate(zip(b_preds, b_true, b_conf_list)):
    if pred == 1 and conf > 0.7:
        xai_idx = i
        break

if xai_idx is None:
    xai_idx = next((i for i, p in enumerate(b_preds) if p == 1), None)

if xai_idx is not None:
    plot_attention_weights(
        b_attn_weights_list[xai_idx],
        predicted_class=BEHAVIOR_CLASSES[b_preds[xai_idx]],
        confidence=b_conf_list[xai_idx],
    )
else:
    print('No shoplifting predictions found in test set - skipping XAI plot.')

C:\Users\johnf\AppData\Local\Temp\ipykernel_19488\3517619641.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  bckpt = torch.load(BILSTM_SAVE, map_location=device)


BiLSTM test split loaded: 40 sequences
BiLSTM loaded (best val acc: 99.8%)


Evaluating BiLSTM: 100%|██████████| 3/3 [00:00<00:00,  5.64it/s]


  BILSTM — TEST SET EVALUATION
  Test sequences : 40
  Accuracy       : 95.00%
  Precision      : 95.45%
  Recall         : 95.00%
  F1 Score       : 94.99%
              precision    recall  f1-score   support

      normal       0.91      1.00      0.95        20
 shoplifting       1.00      0.90      0.95        20

    accuracy                           0.95        40
   macro avg       0.95      0.95      0.95        40
weighted avg       0.95      0.95      0.95        40



<Figure size 600x500 with 2 Axes>

BiLSTM metrics saved to: D:\Santosh\Project_DigitalWitness\models\bilstm_eval.json


<Figure size 1200x400 with 1 Axes>

XAI attention plot saved to: D:\Santosh\Project_DigitalWitness\outputs\cases\attention_xai_example.png


### XAI Interpretation

The bar chart above shows the temporal attention weights produced by the BiLSTM for a single 45-frame (7.5-second) window classified as shoplifting. Each bar height represents the proportion of the model's decision attributed to that specific frame. Frames with weights above the uniform baseline (dashed line) indicate moments of heightened suspicious activity — typically the frames in which concealment gestures or product interactions occurred. This mechanism provides *temporal explainability*: rather than simply reporting a binary classification, the system identifies **when** in the sequence the suspicious behaviour was most prominent, enabling operators to review only the flagged time segment rather than the entire video.